In [93]:
!git clone https://github.com/floCarlee/Uganda_Food_Price_Prediction.git

fatal: destination path 'Uganda_Food_Price_Prediction' already exists and is not an empty directory.


In [94]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os




In [95]:
import pandas as pd

raw_url = "https://raw.githubusercontent.com/floCarlee/Uganda_Food_Price_Prediction/main/data/raw/wfp_food_prices_uga.csv"

df = pd.read_csv(raw_url)

df.head()

,date,admin1,admin2,market,market_id,latitude,longitude,category,commodity,commodity_id,unit,priceflag,pricetype,currency,price,usdprice
0,2006-01-15,Busia,Samia-bugwe,Busia,2665,0.47,34.09,cereals and tubers,Maize,51,KG,actual,Wholesale,UGX,298.40,0.17
1,2006-01-15,Kampala,Central Kampala,Owino,258,0.32,32.57,cereals and tubers,Maize,51,KG,actual,Wholesale,UGX,345.42,0.20
2,2006-01-15,Kampala,Central Kampala,Owino,258,0.32,32.57,cereals and tubers,Rice,52,KG,actual,Wholesale,UGX,1013.40,0.58
3,2006-01-15,Lira,Lira Municipality,Lira,263,2.23,32.91,cereals and tubers,Maize,51,KG,actual,Wholesale,UGX,247.76,0.14
4,2006-02-15,Busia,Samia-bugwe,Busia,2665,0.47,34.09,cereals and tubers,Maize,51,KG,actual,Wholesale,UGX,296.59,0.17


In [96]:
# Load the excel file
# file_path = "wfp_food_prices_uga.csv"
# df = pd.read_csv(file_path)
# df.head()

In [97]:
df.shape


(32171, 16)

In [98]:
df.columns.tolist()

['date',
 'admin1',
 'admin2',
 'market',
 'market_id',
 'latitude',
 'longitude',
 'category',
 'commodity',
 'commodity_id',
 'unit',
 'priceflag',
 'pricetype',
 'currency',
 'price',
 'usdprice']

In [99]:
df.isna().sum()

,0
date,0
admin1,0
admin2,0
market,0
market_id,0
latitude,0
longitude,0
category,0
commodity,0
commodity_id,0


In [100]:
df.duplicated().sum()

np.int64(0)

In [101]:
df["category"].value_counts()

,count
category,
cereals and tubers,14821
non-food,7838
pulses and nuts,3188
oil and fats,2738
vegetables and fruits,1164
miscellaneous food,869
"meat, fish and eggs",802
milk and dairy,751


In [102]:
df["commodity"].value_counts()

,count
commodity,
Beans,3188
Maize flour,3045
Maize (white),2999
Oil (vegetable),2738
Sorghum,2658
Cassava flour,2003
Millet,1665
Salt,869
Millet flour,804


In [103]:
df["market"].value_counts()

,count
market,
Owino,2096
Lira,1780
Kyangwali (refugee settlement),1393
Kyaka II (refugee settlement),1352
Bidibidi (refugee settlement),1338
Lobule (refugee settlement),1308
Adjumani (refugee settlement),1305
Imvepi (refugee settlement),1304
Palorinya (refugee settlement),1294


In [104]:
df["date"].min(), df["date"].max()

('2006-01-15', '2026-06-15')

In [105]:
# Make a copy of the raw dataset so that the original dataset stays unchanged
clean_df = df.copy()
clean_df.head()


,date,admin1,admin2,market,market_id,latitude,longitude,category,commodity,commodity_id,unit,priceflag,pricetype,currency,price,usdprice
0,2006-01-15,Busia,Samia-bugwe,Busia,2665,0.47,34.09,cereals and tubers,Maize,51,KG,actual,Wholesale,UGX,298.40,0.17
1,2006-01-15,Kampala,Central Kampala,Owino,258,0.32,32.57,cereals and tubers,Maize,51,KG,actual,Wholesale,UGX,345.42,0.20
2,2006-01-15,Kampala,Central Kampala,Owino,258,0.32,32.57,cereals and tubers,Rice,52,KG,actual,Wholesale,UGX,1013.40,0.58
3,2006-01-15,Lira,Lira Municipality,Lira,263,2.23,32.91,cereals and tubers,Maize,51,KG,actual,Wholesale,UGX,247.76,0.14
4,2006-02-15,Busia,Samia-bugwe,Busia,2665,0.47,34.09,cereals and tubers,Maize,51,KG,actual,Wholesale,UGX,296.59,0.17


In [106]:
# making the columns consistent
clean_df.columns = clean_df.columns.str.strip().str.lower().str.replace(" ","_")
clean_df.columns.to_list()

['date',
 'admin1',
 'admin2',
 'market',
 'market_id',
 'latitude',
 'longitude',
 'category',
 'commodity',
 'commodity_id',
 'unit',
 'priceflag',
 'pricetype',
 'currency',
 'price',
 'usdprice']

In [107]:
# converting date columns from text to datetime format. this will help to strip the month, date and year
clean_df["date"] = pd.to_datetime(clean_df["date"])
clean_df["date"].dtype



dtype('<M8[ns]')

In [108]:
clean_df["date"].min(), clean_df["date"].max()

(Timestamp('2006-01-15 00:00:00'), Timestamp('2026-06-15 00:00:00'))

In [109]:
# converting the prices to numeric 
clean_df["price"] = pd.to_numeric(clean_df["price"], errors="coerce")
clean_df["usdprice"] = pd.to_numeric(clean_df["usdprice"], errors="coerce")
clean_df[["price", "usdprice"]].dtypes

,0
price,float64
usdprice,float64


In [110]:
# check for missing values  
clean_df[["price", "usdprice"]].isna().sum()

,0
price,0
usdprice,0


In [111]:
# check for invalid prices
clean_df[clean_df["price"] <= 0].shape

(0, 16)

In [112]:
#checking for duplicates
clean_df.duplicated().sum()
clean_df.shape

(32171, 16)

In [113]:
#clean text columns

text_columns = clean_df.select_dtypes(include="object").columns

for col in text_columns:
    clean_df[col] = clean_df[col].astype(str).str.strip()

clean_df.head()

,date,admin1,admin2,market,market_id,latitude,longitude,category,commodity,commodity_id,unit,priceflag,pricetype,currency,price,usdprice
0,2006-01-15,Busia,Samia-bugwe,Busia,2665,0.47,34.09,cereals and tubers,Maize,51,KG,actual,Wholesale,UGX,298.40,0.17
1,2006-01-15,Kampala,Central Kampala,Owino,258,0.32,32.57,cereals and tubers,Maize,51,KG,actual,Wholesale,UGX,345.42,0.20
2,2006-01-15,Kampala,Central Kampala,Owino,258,0.32,32.57,cereals and tubers,Rice,52,KG,actual,Wholesale,UGX,1013.40,0.58
3,2006-01-15,Lira,Lira Municipality,Lira,263,2.23,32.91,cereals and tubers,Maize,51,KG,actual,Wholesale,UGX,247.76,0.14
4,2006-02-15,Busia,Samia-bugwe,Busia,2665,0.47,34.09,cereals and tubers,Maize,51,KG,actual,Wholesale,UGX,296.59,0.17


In [114]:
# Filtering for the retail price for food
food_retail_df = clean_df[
    (clean_df["pricetype"].str.lower() == "retail") &
    (clean_df["category"].str.lower() != "non-food")
].copy()

food_retail_df.shape


(22948, 16)

In [115]:
food_retail_df["category"].value_counts()

,count
category,
cereals and tubers,13940
pulses and nuts,2879
oil and fats,2738
vegetables and fruits,969
miscellaneous food,869
"meat, fish and eggs",802
milk and dairy,751


In [116]:
# creating time based features
food_retail_df["year"] = food_retail_df["date"].dt.year
food_retail_df["month"] = food_retail_df["date"].dt.month
food_retail_df["quarter"] = food_retail_df["date"].dt.quarter

food_retail_df[["date", "year", "month", "quarter"]].head()

,date,year,month,quarter
101,2008-07-15,2008,7,3
105,2008-07-15,2008,7,3
110,2008-08-15,2008,8,3
114,2008-08-15,2008,8,3
119,2008-09-15,2008,9,3


In [117]:
# sorting by commodity, market and date
food_retail_df = food_retail_df.sort_values(
    by=["commodity_id", "market_id", "date"]
).copy()

food_retail_df.head()

,date,admin1,admin2,market,market_id,latitude,longitude,category,commodity,commodity_id,unit,priceflag,pricetype,currency,price,usdprice,year,month,quarter
661,2011-04-15,Kyankwanzi,Kiboga,Kiboga,257,0.92,31.77,pulses and nuts,Beans,50,KG,actual,Retail,UGX,1950.0,0.82,2011,4,2
713,2011-05-15,Kyankwanzi,Kiboga,Kiboga,257,0.92,31.77,pulses and nuts,Beans,50,KG,actual,Retail,UGX,2150.0,0.90,2011,5,2
763,2011-06-15,Kyankwanzi,Kiboga,Kiboga,257,0.92,31.77,pulses and nuts,Beans,50,KG,actual,Retail,UGX,2000.0,0.82,2011,6,2
816,2011-07-15,Kyankwanzi,Kiboga,Kiboga,257,0.92,31.77,pulses and nuts,Beans,50,KG,actual,Retail,UGX,1500.0,0.59,2011,7,3
868,2011-08-15,Kyankwanzi,Kiboga,Kiboga,257,0.92,31.77,pulses and nuts,Beans,50,KG,actual,Retail,UGX,1667.0,0.60,2011,8,3


In [118]:
# create historical features so that the model can learn from previous prices
food_retail_df["previous_price"] = food_retail_df.groupby(
    ["commodity_id", "market_id"]
)["price"].shift(1)

food_retail_df["price_change"] = food_retail_df["price"] - food_retail_df["previous_price"]

food_retail_df["price_pct_change"] = (
    food_retail_df["price_change"] / food_retail_df["previous_price"]
)

food_retail_df[["date", "market", "commodity", "price", "previous_price", "price_change", "price_pct_change"]].head(10)

,date,market,commodity,price,previous_price,price_change,price_pct_change
661,2011-04-15,Kiboga,Beans,1950.0,NaN,NaN,NaN
713,2011-05-15,Kiboga,Beans,2150.0,1950.0,200.0,0.102564
763,2011-06-15,Kiboga,Beans,2000.0,2150.0,-150.0,-0.069767
816,2011-07-15,Kiboga,Beans,1500.0,2000.0,-500.0,-0.250000
868,2011-08-15,Kiboga,Beans,1667.0,1500.0,167.0,0.111333
917,2011-09-15,Kiboga,Beans,1500.0,1667.0,-167.0,-0.100180
969,2011-10-15,Kiboga,Beans,1900.0,1500.0,400.0,0.266667
1023,2011-11-15,Kiboga,Beans,1632.0,1900.0,-268.0,-0.141053
1120,2012-01-15,Kiboga,Beans,1425.0,1632.0,-207.0,-0.126838
1173,2012-02-15,Kiboga,Beans,1450.0,1425.0,25.0,0.017544


In [119]:
# create moving average features
food_retail_df["price_3_month_avg"] = food_retail_df.groupby(
    ["commodity_id", "market_id"]
)["price"].transform(lambda x: x.rolling(window=3).mean())

food_retail_df["price_6_month_avg"] = food_retail_df.groupby(
    ["commodity_id", "market_id"]
)["price"].transform(lambda x: x.rolling(window=6).mean())

food_retail_df[[
    "date", "market", "commodity", "price",
    "price_3_month_avg", "price_6_month_avg"
]].head(10)

,date,market,commodity,price,price_3_month_avg,price_6_month_avg
661,2011-04-15,Kiboga,Beans,1950.0,NaN,NaN
713,2011-05-15,Kiboga,Beans,2150.0,NaN,NaN
763,2011-06-15,Kiboga,Beans,2000.0,2033.333333,NaN
816,2011-07-15,Kiboga,Beans,1500.0,1883.333333,NaN
868,2011-08-15,Kiboga,Beans,1667.0,1722.333333,NaN
917,2011-09-15,Kiboga,Beans,1500.0,1555.666667,1794.500000
969,2011-10-15,Kiboga,Beans,1900.0,1689.000000,1786.166667
1023,2011-11-15,Kiboga,Beans,1632.0,1677.333333,1699.833333
1120,2012-01-15,Kiboga,Beans,1425.0,1652.333333,1604.000000
1173,2012-02-15,Kiboga,Beans,1450.0,1502.333333,1595.666667


In [120]:
# creating the target ie next month price
food_retail_df["next_month_price"] = food_retail_df.groupby(
    ["commodity_id", "market_id"]
)["price"].shift(-1)

food_retail_df[["date", "market", "commodity", "price", "next_month_price"]].head(10)

,date,market,commodity,price,next_month_price
661,2011-04-15,Kiboga,Beans,1950.0,2150.0
713,2011-05-15,Kiboga,Beans,2150.0,2000.0
763,2011-06-15,Kiboga,Beans,2000.0,1500.0
816,2011-07-15,Kiboga,Beans,1500.0,1667.0
868,2011-08-15,Kiboga,Beans,1667.0,1500.0
917,2011-09-15,Kiboga,Beans,1500.0,1900.0
969,2011-10-15,Kiboga,Beans,1900.0,1632.0
1023,2011-11-15,Kiboga,Beans,1632.0,1425.0
1120,2012-01-15,Kiboga,Beans,1425.0,1450.0
1173,2012-02-15,Kiboga,Beans,1450.0,1609.0


In [121]:
# check and remove the missing records created by the new columns
food_retail_df.isna().sum()


,0
date,0
admin1,0
admin2,0
market,0
market_id,0
latitude,0
longitude,0
category,0
commodity,0
commodity_id,0


In [122]:
# remove the duplicate values
ml_df = food_retail_df.dropna(subset=[
    "previous_price",
    "price_change",
    "price_pct_change",
    "price_3_month_avg",
    "price_6_month_avg",
    "next_month_price"
]).copy()

ml_df.shape

(20872, 25)

In [123]:
# check and remove the missing records created by the new columns
ml_df.isna().sum()

,0
date,0
admin1,0
admin2,0
market,0
market_id,0
latitude,0
longitude,0
category,0
commodity,0
commodity_id,0


In [124]:
# selecting final text_columns
model_columns = [
    "date",
    "year",
    "month",
    "quarter",
    "admin1",
    "admin2",
    "market",
    "market_id",
    "latitude",
    "longitude",
    "category",
    "commodity",
    "commodity_id",
    "unit",
    "price",
    "previous_price",
    "price_change",
    "price_pct_change",
    "price_3_month_avg",
    "price_6_month_avg",
    "next_month_price"
]

ml_df = ml_df[model_columns].copy()

ml_df.head()

,date,year,month,quarter,admin1,admin2,market,market_id,latitude,longitude,...,commodity,commodity_id,unit,price,previous_price,price_change,price_pct_change,price_3_month_avg,price_6_month_avg,next_month_price
917,2011-09-15,2011,9,3,Kyankwanzi,Kiboga,Kiboga,257,0.92,31.77,...,Beans,50,KG,1500.0,1667.0,-167.0,-0.100180,1555.666667,1794.500000,1900.0
969,2011-10-15,2011,10,4,Kyankwanzi,Kiboga,Kiboga,257,0.92,31.77,...,Beans,50,KG,1900.0,1500.0,400.0,0.266667,1689.000000,1786.166667,1632.0
1023,2011-11-15,2011,11,4,Kyankwanzi,Kiboga,Kiboga,257,0.92,31.77,...,Beans,50,KG,1632.0,1900.0,-268.0,-0.141053,1677.333333,1699.833333,1425.0
1120,2012-01-15,2012,1,1,Kyankwanzi,Kiboga,Kiboga,257,0.92,31.77,...,Beans,50,KG,1425.0,1632.0,-207.0,-0.126838,1652.333333,1604.000000,1450.0
1173,2012-02-15,2012,2,1,Kyankwanzi,Kiboga,Kiboga,257,0.92,31.77,...,Beans,50,KG,1450.0,1425.0,25.0,0.017544,1502.333333,1595.666667,1609.0


In [125]:
ml_df.shape

(20872, 21)

In [126]:
ml_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20872 entries, 917 to 31562
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   date               20872 non-null  datetime64[ns]
 1   year               20872 non-null  int32         
 2   month              20872 non-null  int32         
 3   quarter            20872 non-null  int32         
 4   admin1             20872 non-null  object        
 5   admin2             20872 non-null  object        
 6   market             20872 non-null  object        
 7   market_id          20872 non-null  int64         
 8   latitude           20872 non-null  float64       
 9   longitude          20872 non-null  float64       
 10  category           20872 non-null  object        
 11  commodity          20872 non-null  object        
 12  commodity_id       20872 non-null  int64         
 13  unit               20872 non-null  object        
 14  price    

In [127]:
ml_df.describe()

,date,year,month,quarter,market_id,latitude,longitude,commodity_id,price,previous_price,price_change,price_pct_change,price_3_month_avg,price_6_month_avg,next_month_price
count,20872,20872.000000,20872.000000,20872.00000,20872.000000,20872.000000,20872.000000,20872.000000,20872.000000,20872.000000,20872.000000,20872.000000,20872.000000,20872.000000,20872.000000
mean,2022-01-14 04:47:21.088539904,2021.537610,6.549300,2.51811,3779.481075,1.592159,32.301729,160.445765,3441.605277,3428.384796,13.220482,0.049365,3428.826130,3410.069793,3453.639590
min,2008-12-15 00:00:00,2008.000000,1.000000,1.00000,257.000000,-0.940000,30.070000,50.000000,2.000000,2.000000,-44869.000000,-0.998363,285.133333,308.900000,2.000000
25%,2021-03-15 00:00:00,2021.000000,4.000000,2.00000,263.000000,0.360000,31.090000,67.000000,1457.750000,1447.000000,-110.000000,-0.050000,1456.666667,1450.000000,1467.000000
50%,2023-04-15 00:00:00,2023.000000,7.000000,3.00000,6148.000000,1.690000,32.310000,74.000000,2044.500000,2020.500000,0.000000,0.000000,2074.333333,2069.635833,2067.000000
75%,2024-09-15 00:00:00,2024.000000,10.000000,4.00000,6410.000000,3.000000,33.470000,96.000000,3730.750000,3712.250000,125.000000,0.063019,3682.833333,3637.583333,3750.000000
max,2026-05-15 00:00:00,2026.000000,12.000000,4.00000,7957.000000,3.570000,34.830000,900.000000,66608.000000,66608.000000,25907.000000,542.000000,43227.000000,33750.166667,66608.000000
std,NaN,3.849129,3.428925,1.11069,2864.774226,1.404049,1.317002,222.995054,4119.322213,4105.269851,1226.391193,3.758770,4039.309957,3991.898200,4134.444675


In [128]:
# save the final clean dataset
ml_df.to_csv("/content/wfp_uganda_food_prices_ml_ready.csv", index=False)




print("ETL completed successfully.")

ETL completed successfully.


In [129]:
pd.read_csv("/content/wfp_uganda_food_prices_ml_ready.csv").head()

,date,year,month,quarter,admin1,admin2,market,market_id,latitude,longitude,...,commodity,commodity_id,unit,price,previous_price,price_change,price_pct_change,price_3_month_avg,price_6_month_avg,next_month_price
0,2011-09-15,2011,9,3,Kyankwanzi,Kiboga,Kiboga,257,0.92,31.77,...,Beans,50,KG,1500.0,1667.0,-167.0,-0.100180,1555.666667,1794.500000,1900.0
1,2011-10-15,2011,10,4,Kyankwanzi,Kiboga,Kiboga,257,0.92,31.77,...,Beans,50,KG,1900.0,1500.0,400.0,0.266667,1689.000000,1786.166667,1632.0
2,2011-11-15,2011,11,4,Kyankwanzi,Kiboga,Kiboga,257,0.92,31.77,...,Beans,50,KG,1632.0,1900.0,-268.0,-0.141053,1677.333333,1699.833333,1425.0
3,2012-01-15,2012,1,1,Kyankwanzi,Kiboga,Kiboga,257,0.92,31.77,...,Beans,50,KG,1425.0,1632.0,-207.0,-0.126838,1652.333333,1604.000000,1450.0
4,2012-02-15,2012,2,1,Kyankwanzi,Kiboga,Kiboga,257,0.92,31.77,...,Beans,50,KG,1450.0,1425.0,25.0,0.017544,1502.333333,1595.666667,1609.0


In [130]:
# download it to the device
from google.colab import files

files.download("/content/wfp_uganda_food_prices_ml_ready.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>